## observation data 

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
import matplotlib as mpl
import matplotlib.ticker as mticker
import netCDF4
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [3]:
from scipy import stats

In [4]:
mpl.rcParams['font.family'] = 'Droid Sans'
mpl.rcParams['font.size'] = 12
# Edit axes parameters
mpl.rcParams['axes.linewidth'] = 1.5
# Tick properties
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.minor.size'] = 3
mpl.rcParams['xtick.major.width'] = 1
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.minor.size'] = 3
mpl.rcParams['ytick.major.width'] = 1
mpl.rcParams['ytick.direction'] = 'out'

In [5]:
# plt.style.use('dark_background')

### Functions needed for the analysis

In [6]:
import matplotlib as m
from matplotlib.colors import BoundaryNorm as BM
import matplotlib.patches as mpatches

def plot_background(ax):
    ax.add_feature(cfeature.COASTLINE, alpha=0.9, lw=1.1)
    ax.set_global()
    # ax.add_feature(cfeature.LAND, color='lightgray')
    # ax.add_feature(cfeature.OCEAN, color='lightgray')
    gl = ax.gridlines(draw_labels=True,
                      linewidth=1, color='gray', alpha=0.01, linestyle='--')
    gl.top_labels = False
    # gl.left_labels = False
    # gl.bottom_labels = False
    gl.right_labels = False
    gl.xlines = False
    # gl.xlocator = mticker.FixedLocator([-180, -45, 0, 45, 180])
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 10, 'color': 'k'}
    gl.ylabel_style = {'size': 10, 'color': 'k'}
    return ax


def plot_maps(x, y, z, titles, labels, cmap, levels, cbar_label = 'Precip', pval = [], nrows=1, ncols=3, figsize=(12,4), land_mask_list = [0], add_patch=False):
    fig, axarr = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, subplot_kw={'projection':ccrs.Robinson(central_longitude=180)})
    
    axlist = axarr.flatten()
    
    for ax in axlist:
        plot_background(ax)
    
    for i in range(len(z)):
        axlist[i].contourf(x, y, z[i], cmap = cmap, transform = ccrs.PlateCarree(central_longitude=0), levels=levels, extend='both')
        axlist[i].set_title(titles[i])
        if i in land_mask_list:
            axlist[i].add_feature(cfeature.LAND, color = 'k', zorder=1)
        if pval != []:
            pval_plot = np.ma.masked_less_equal(pval[i], 0.05)
            axlist[i].pcolor(x, y, pval_plot, alpha = 0., hatch='////', transform = ccrs.PlateCarree(central_longitude=0))
        axlist[i].set_title(titles[i], fontdict={'fontsize':12})
        axlist[i].text(0.1, 1.05, labels[i], size=16, fontweight='bold', transform=axlist[i].transAxes)
        if add_patch:
            axlist[i].add_patch(mpatches.Rectangle(xy=[120, -65], width=170, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[190, -5], width=80, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[140, -5], width=30, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[250, -30], width=40, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
        
    norm = BM(levels, 256, extend='both')
    fig.colorbar(m.cm.ScalarMappable(norm = norm, cmap=cmap), ax = axlist, \
                orientation = 'horizontal', shrink=0.4, aspect = 20, pad = 0.05, label = cbar_label)

In [7]:
from functions import preproc_funcs as funcs

In [8]:
from functions import xr_lowess

In [9]:
from statsmodels.tsa.seasonal import STL


def loess1d(x, period):
    x_copy = x.copy()
    res = STL(x_copy, period=period).fit()
    return res.trend


def loess3d(x, dim, period):
    return xr.apply_ufunc(loess1d, x, input_core_dims=[[dim]], output_core_dims=[[dim]], kwargs=dict(period=period), vectorize=True, dask="parallelized")

In [14]:
import glob
import multiprocessing as mp

In [15]:
# Function to find the first file in each model's r1* directory
def find_all_files(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[7]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files


In [16]:
psl_pattern_hist = '/g/data/lp01/CMIP6/CMIP/*/*/historical/r1i1*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp5 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp585/r1i1*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp3 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp370/r1i1*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp2 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp245/r1i1*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp1 = '/g/data/lp01/CMIP6/ScenarioMIP/*/*/ssp126/r1i1*/Amon/psl/gr1.5/*/*.nc'


In [17]:
psl_files_hist = find_all_files(psl_pattern_hist)
psl_files_ssp5 = find_all_files(psl_pattern_ssp5)
psl_files_ssp3 = find_all_files(psl_pattern_ssp3)
psl_files_ssp2 = find_all_files(psl_pattern_ssp2)
psl_files_ssp1 = find_all_files(psl_pattern_ssp1)

In [19]:
paths = ['/g/data/fs38/publications/CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp534-over/r1i1p1f1/Amon/psl/gn/v20210928/psl_Amon_ACCESS-CM2_ssp534-over_r1i1p1f1_gn_204101-210012.nc',
'/g/data/fs38/publications/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp534-over/r1i1p1f1/Amon/psl/gn/v20211020/psl_Amon_ACCESS-ESM1-5_ssp534-over_r1i1p1f1_gn_204101-210012.nc',
'/g/data/oi10/replicas/CMIP6/ScenarioMIP/CAS/FGOALS-g3/ssp534-over/r1i1p1f1/Amon/psl/gn/v20200410/*.nc',
'/g/data/oi10/replicas/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp534-over/r1i1p1f1/Amon/psl/gn/v20190429/*.nc',
'/g/data/oi10/replicas/CMIP6/ScenarioMIP/IPSL/IPSL-CM6A-LR/ssp534-over/r1i1p1f1/Amon/psl/gr/v20190909/*.nc',
'/g/data/oi10/replicas/CMIP6/ScenarioMIP/MIROC/MIROC6/ssp534-over/r1i1p1f1/Amon/psl/gn/v20190807/psl_Amon_MIROC6_ssp534-over_r1i1p1f1_gn_204001-210012.nc',
'/g/data/oi10/replicas/CMIP6/ScenarioMIP/MRI/MRI-ESM2-0/ssp534-over/r1i1p1f1/Amon/psl/gn/v20191108/*.nc',]
model_names = ['ACCESS-CM2', 'ACCESS-ESM1-5', 'FGOALS-g3', 'CanESM5', 'IPSL-CM6A-LR', 'MIROC6', 'MRI-ESM2-0']
psl_files_ssp5_over = dict(zip(model_names, paths))
psl_files_ssp5_over

{'ACCESS-CM2': '/g/data/fs38/publications/CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp534-over/r1i1p1f1/Amon/psl/gn/v20210928/psl_Amon_ACCESS-CM2_ssp534-over_r1i1p1f1_gn_204101-210012.nc',
 'ACCESS-ESM1-5': '/g/data/fs38/publications/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp534-over/r1i1p1f1/Amon/psl/gn/v20211020/psl_Amon_ACCESS-ESM1-5_ssp534-over_r1i1p1f1_gn_204101-210012.nc',
 'FGOALS-g3': '/g/data/oi10/replicas/CMIP6/ScenarioMIP/CAS/FGOALS-g3/ssp534-over/r1i1p1f1/Amon/psl/gn/v20200410/*.nc',
 'CanESM5': '/g/data/oi10/replicas/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp534-over/r1i1p1f1/Amon/psl/gn/v20190429/*.nc',
 'IPSL-CM6A-LR': '/g/data/oi10/replicas/CMIP6/ScenarioMIP/IPSL/IPSL-CM6A-LR/ssp534-over/r1i1p1f1/Amon/psl/gr/v20190909/*.nc',
 'MIROC6': '/g/data/oi10/replicas/CMIP6/ScenarioMIP/MIROC/MIROC6/ssp534-over/r1i1p1f1/Amon/psl/gn/v20190807/psl_Amon_MIROC6_ssp534-over_r1i1p1f1_gn_204001-210012.nc',
 'MRI-ESM2-0': '/g/data/oi10/replicas/CMIP6/ScenarioMIP/MRI/MRI-ESM2-0/ssp534-over/r1i1p1

In [20]:
# ts_pattern_access_hist = '/g/data/ob22/as8561/data/enso_trans_stable/hist_ssp_runs/regrid_sst/'
# ts_pattern_access_ssp = '/g/data/ob22/as8561/data/enso_trans_stable/hist_ssp_runs/regrid_sst'

In [21]:
import xesmf as xe

In [22]:
# temp_hist = xr.open_dataset(ts_files_hist['ACCESS-CM2'])
# temp_hist

In [23]:
ds_out = xe.util.cf_grid_2d(-0.75, 360, 1.5, -90, 90, 1.5)
ds_out

<xarray.Dataset>
Dimensions:             (lon: 240, bound: 2, lat: 120)
Coordinates:
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
    latitude_longitude  float64 nan
Dimensions without coordinates: bound
Data variables:
    lon_bounds          (lon, bound) float64 -0.75 0.75 0.75 ... 357.8 359.2
    lat_bounds          (lat, bound) float64 -90.0 -88.5 -88.5 ... 88.5 90.0

In [114]:
# Function to process a single model and return the detrended NINO3.4 and precip anomalies
def process_model(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasets
        var_file_hist = psl_files_hist[model_identifier]
        var_file_ssp = psl_files_ssp3[model_identifier]
        ds_var_hist = xr.open_mfdataset(var_file_hist, use_cftime=True).sel(time = slice('1850-01-01', '2015-01-01'))
        ds_var_ssp = xr.open_mfdataset(var_file_ssp, use_cftime=True)
        # add custom time ranges
        ds_var_hist['time'] = xr.cftime_range('1850-01-01', '2015-01-01', freq='1M')
        ssp_end_year = int(ds_var_ssp.time.dt.year[-1])
        ds_var_ssp['time'] = xr.cftime_range('2015-01-01', f'{ssp_end_year + 1}-01-01', freq='1M')
        #
        var = xr.concat([ds_var_hist, ds_var_ssp], dim='time').psl.resample(time = 'AS-JUN').mean('time').load()  # var data
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        var_anom = funcs.calc_anom_annual(var, var.sel(time = slice('1960', '1990')))
        # var_trend = funcs.calc_trend3d(var_anom.sel(time = slice('1980', '2014')), 'time')
        # var_trend_pval = funcs.calc_trend_pval3d(var_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        weights = np.cos(np.deg2rad(var.lat))
        wp_var = var_anom.sel(lat = slice(-5,5), lon = slice(140, 170)).weighted(weights).mean(('lat', 'lon'))
        ct_var = var_anom.sel(lat = slice(-5,5), lon = slice(190, 270)).weighted(weights).mean(('lat', 'lon'))

        # print(f'Completed: {model_identifie}')
        # return model_identifier, var_trend, var_trend_pval, gmst_anom, nino34_index, wp_var, ct_var, so_var
        return model_identifier, wp_var - ct_var
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")
        # break



# Function to process a single model and return the detrended NINO3.4 and precip anomalies
# TODO : remapping required for ssp534_over
def process_model_overshoot(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasets
        var_file_hist = psl_files_hist[model_identifier]
        var_file_ssp5_over = psl_files_ssp5_over[model_identifier]
        var_file_ssp5 = psl_files_ssp5[model_identifier]
        ds_var_hist = xr.open_mfdataset(var_file_hist, use_cftime=True).sel(time = slice('1850-01-01', '2015-01-01'))
        ds_var_ssp5_over = xr.open_mfdataset(var_file_ssp5_over, use_cftime=True).load()
        regridder = xe.Regridder(ds_var_ssp5_over, ds_out, 'bilinear', periodic=True)
        ds_var_ssp5_over_regr = regridder(ds_var_ssp5_over)
        ds_var_ssp5 = xr.open_mfdataset(var_file_ssp5)
        # add custom time ranges
        ds_var_hist['time'] = xr.cftime_range('1850-01-01', '2015-01-01', freq='1M')
        ssp_start_year = int(ds_var_ssp5_over_regr.time.dt.year[0])
        ssp_end_year = int(ds_var_ssp5_over_regr.time.dt.year[-1])
        ds_var_ssp5 = ds_var_ssp5.sel(time = slice(str(2015), str(ssp_start_year-1)))
        ds_var_ssp5['time'] = xr.cftime_range('2015-01-01', f'{ssp_start_year}-01-01', freq='1M')
        ds_var_ssp5_over_regr['time'] = xr.cftime_range(f'{ssp_start_year}-01-01', f'{ssp_end_year + 1}-01-01', freq='1M')
        
        var = xr.concat([ds_var_hist, ds_var_ssp5, ds_var_ssp5_over_regr], dim='time').psl.resample(time = 'AS-JUN').mean('time').load()  # var data
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        var_anom = funcs.calc_anom_annual(var, var.sel(time = slice('1960', '1990')))
        # # var_trend = funcs.calc_trend3d(var_anom.sel(time = slice('1980', '2014')), 'time')
        # var_trend_pval = funcs.calc_trend_pval3d(var_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        weights = np.cos(np.deg2rad(var.lat))
        wp_var = var_anom.sel(lat = slice(-5,5), lon = slice(140, 170)).weighted(weights).mean(('lat', 'lon'))
        ct_var = var_anom.sel(lat = slice(-5,5), lon = slice(190, 270)).weighted(weights).mean(('lat', 'lon'))

        # print(f'Completed: {model_identifie}')
        return model_identifier, wp_var - ct_var
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")
        # break



In [115]:
models_to_process = [(model) for model in psl_files_hist if model in psl_files_ssp5_over and model != 'IPSL-CM5A2-INCA']
len(models_to_process), models_to_process

(7,
 ['CanESM5',
  'MIROC6',
  'IPSL-CM6A-LR',
  'ACCESS-CM2',
  'MRI-ESM2-0',
  'FGOALS-g3',
  'ACCESS-ESM1-5'])

In [116]:
# Run multiprocessing and gather results
res_arr = []
with mp.Pool(processes=mp.cpu_count()) as pool:
    i = 0
    for res in pool.imap(process_model_overshoot, models_to_process):
        res_arr.append(res)
        print(f'Completed {i+1}/{len(models_to_process)}', end='\r')
        i += 1



In [117]:
# models = np.array(res_arr)[:, 0]
# trend = np.array(res_arr)[:, 1]
# trend_pval = np.array(res_arr)[:, 2]
# nino34_index = np.array(res_arr)[:, 3]
# wp_sst = np.array(res_arr)[:, 4]
# cp_sst = np.array(res_arr)[:, 5]
# so_sst = np.array(res_arr)[:, 6]

In [118]:
def process_extensions(arr, get_extensions=True):
    if get_extensions:
        return [da for da in arr if len(da.time) == 452]
    else:
        return [da.sel(time = slice('1850-01-01', '2099-12-01')) for da in arr]

In [119]:
model_list = np.array(res_arr)[:, 0]

In [120]:
np.array(res_arr).shape

(7, 2)

In [121]:
model_list

array(['CanESM5', 'MIROC6', 'IPSL-CM6A-LR', 'ACCESS-CM2', 'MRI-ESM2-0',
       'FGOALS-g3', 'ACCESS-ESM1-5'], dtype=object)

In [122]:
test = np.array(res_arr)[:, 1]
model_extensions = []
for i in range(len(test)):
    da = test[i]
    # print(len(da.time))
    if len(da.time) == 452:
        model_extensions.append(model_list[i])



model_extensions

['CanESM5', 'IPSL-CM6A-LR', 'MRI-ESM2-0']

In [124]:
len(process_extensions(np.array(res_arr)[:, 1], get_extensions=False))

7

In [125]:
# model_trend = xr.concat(np.array(res_arr)[:, 1], dim=model_list).rename(dict(concat_dim = 'model')).to_dataset(name = 'trend')
# model_pval = xr.concat(np.array(res_arr)[:, 2], dim=model_list).rename(dict(concat_dim = 'model')).to_dataset(name = 'pval')


In [126]:
model_wc_index = xr.concat(process_extensions(np.array(res_arr)[:, 1], get_extensions=False), dim=model_list).rename(dict(concat_dim = 'model')).to_dataset(name = 'wc_index')

In [127]:
out = xr.merge([model_wc_index])
out

<xarray.Dataset>
Dimensions:             (time: 250, model: 7)
Coordinates:
  * time                (time) object 1850-06-01 00:00:00 ... 2099-06-01 00:0...
    latitude_longitude  float64 nan
  * model               (model) object 'CanESM5' 'MIROC6' ... 'ACCESS-ESM1-5'
Data variables:
    wc_index            (model, time) float64 -61.16 -10.98 ... 60.79 -1.348

In [128]:
model_wc_index = xr.concat(process_extensions(np.array(res_arr)[:, 1], get_extensions=True), dim=model_extensions).rename(dict(concat_dim = 'model')).to_dataset(name = 'wc_index')

In [129]:
out_extensions = xr.merge([model_wc_index])
out_extensions

<xarray.Dataset>
Dimensions:             (time: 452, model: 3)
Coordinates:
  * time                (time) object 1849-06-01 00:00:00 ... 2300-06-01 00:0...
    latitude_longitude  float64 nan
  * model               (model) object 'CanESM5' 'IPSL-CM6A-LR' 'MRI-ESM2-0'
Data variables:
    wc_index            (model, time) float64 -12.27 -61.16 ... -76.39 36.04

In [130]:
def find_model_agreement1d(arr):
    agree = 0
    for val in arr:
        if val < 0.05:
            agree += 1
    return agree/len(arr)

def find_model_agreement3d(da, dim):
    return xr.apply_ufunc(find_model_agreement1d, da, input_core_dims=[[dim]], vectorize=True, dask='parallelized')

In [131]:
# out['pval_agreement'] = find_model_agreement3d(out.pval, dim='model')
# out_extensions['pval_agreement'] = find_model_agreement3d(out_extensions.pval, dim='model')

In [132]:
out.to_netcdf('./data/res/psl_ssp5_over.nc')
out_extensions.to_netcdf('./data/res/psl_ssp5_over_extensions.nc')